In [ ]:
import rioxarray
import importlib
import xarray as xr
import numpy as np
from scipy.spatial import cKDTree
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
import fourierParam as params

In [ ]:
ndvi_data = rioxarray.open_rasterio('../input/cut_files/ndvi_cut_ex.tif')
vh_data = rioxarray.open_rasterio('../input/cut_files/vh_cut_ex.tif')

input_path = "../input/cut_files/mask/maskLuatom.tif"
mask_data = rioxarray.open_rasterio(input_path)
ndvi_data

In [ ]:
import xarray as xr
import numpy as np

# Hàm dùng để tìm các vị trí của mask

def find_xy_valid(mask_data):
    mask = mask_data != 255  # Boolean mask where values are not 255
    # Stack x and y dimensions into a single index
    data_stacked = mask_data.stack(points=("y", "x"))
    # Apply the mask and drop NaN values
    filtered = data_stacked.where(data_stacked != 255, drop=True)
    # Extract x and y coordinates
    x_indices = filtered.coords["x"].values
    y_indices = filtered.coords["y"].values
    # Stack into a (N, 2) NumPy array
    xy_valid = np.column_stack((x_indices, y_indices))
    print(xy_valid)  # Print or use the filtered coordinates
    return xy_valid

In [ ]:
xy_valid = find_xy_valid(mask_data)

In [ ]:
# Hàm dung để tạo ra chuỗi ảnh giống như ảnh ndvi input
def copy_ndvi(xy_valid, ndvi_data):
    # Tạo DataArray đầu ra với NaN
    ndvi_copy = xr.full_like(ndvi_data, fill_value=np.nan)
    # Lưới toạ độ NDVI
    x_vals = ndvi_data.coords["x"].values
    y_vals = ndvi_data.coords["y"].values
    xx, yy = np.meshgrid(x_vals, y_vals)
    # KDTree từ grid NDVI
    tree = cKDTree(np.column_stack([xx.ravel(), yy.ravel()]))
    # Lặp qua từng band
    for band in ndvi_data.band:
        print(f"Đang xử lý band {band.values}...")
        # NDVI của band hiện tại
        band_data = ndvi_data.sel(band=band)
        # Tìm pixel gần nhất với toàn bộ xy_valid
        _, indices = tree.query(xy_valid)
        # Chuyển index về (y, x)
        y_idx, x_idx = np.unravel_index(indices, xx.shape)
        # Lấy giá trị NDVI tại vị trí đó
        ndvi_values = band_data.values[y_idx, x_idx]
        # Tạo mảng mask với toàn NaN
        band_result = np.full_like(band_data.values, -2., dtype=np.float32)
        # Gán giá trị vào vị trí tương ứng
        band_result[y_idx, x_idx] = ndvi_values
        # Gán vào kết quả tổng
        ndvi_copy.loc[dict(band=band)] = band_result
        return ndvi_copy

In [ ]:
ndvi_copy = copy_ndvi(xy_valid, ndvi_data)

In [ ]:
# Lấy tham số đã tính trước đó
bias = params.lua2v_bias
scales = params.ctxd_scale

In [ ]:
def fourierCal(vh):
    ndvi_preds = []
    fft_vh = fft(vh)
    intercep = fft_vh*scales
    res = np.real(ifft(intercep)) + bias
    ndvi_preds.append(res)
    return ndvi_preds


In [ ]:
# chọn random một vbaif điểm để xem thử kết quả có thể bỏ qua
idx = 100
months = np.arange(13)
ndvi_act = []
ndvi_preds = []
for idx in range(0, 200):
    vh_test = vh_data.sel(x=xy_valid[idx][0], y=xy_valid[idx][1], method="nearest")
    ndvi_test = ndvi_data.sel(x=xy_valid[idx][0], y=xy_valid[idx][1], method="nearest")
    vh_test = np.array(vh_test)
    ndvi_res = fourierCal(vh_test)
    ndvi_act.append(ndvi_test.values)
    ndvi_preds.append(ndvi_res)
    # print(ndvi_test.values)
    # print(ndvi_res)
    plt.plot(months, np.squeeze(ndvi_res))
plt.show()
ndvi_preds = np.squeeze(ndvi_preds)

In [ ]:
def predict_ndvi(xy_valid, vh_data, ndvi_copy):
    for idx in xy_valid:
        vh_val = vh_data.sel(x=idx[0], y=idx[1], method="nearest")
        vh_val = np.array(vh_val)
        ndvi_predict = np.squeeze(fourierCal(vh_val))
        # print(ndvi_predict)
        # Bước 1: Tìm tọa độ gần nhất có trong tập dữ liệu
        nearest_point = ndvi_copy.sel(x=idx[0], y=idx[1], method="nearest")
        nearest_x = float(nearest_point['x'].values)
        nearest_y = float(nearest_point['y'].values)
        # Bước 2: Gán giá trị vào đúng tọa độ gần nhất
        # print("values before filling")
        # print(ndvi_copy.loc[dict(x=nearest_x, y=nearest_y)])
        ndvi_copy.loc[dict(x=nearest_x, y=nearest_y)] = ndvi_predict
        return ndvi_copy

In [ ]:
ndvi_copy = predict_ndvi(xy_valid, vh_data, ndvi_copy)

In [ ]:
alpha = 0.3

def custom_clip_ndvi(ndvi: xr.DataArray) -> xr.DataArray:
    ndvi_corrected = ndvi.copy()
    
    mask_high = ndvi_corrected > 1
    ndvi_corrected = xr.where(mask_high, 1 + alpha * np.log10(ndvi_corrected - 1), ndvi_corrected)
    
    mask_low = ndvi_corrected < -1
    ndvi_corrected = xr.where(mask_low, -1 - alpha * np.log10(-ndvi_corrected - 1), ndvi_corrected)
    
    drop = xr.where(ndvi_corrected<-1, -1, ndvi_corrected)
    drop = xr.where(drop>1, -1, drop)
    return drop

In [ ]:
clipped_ndvi = custom_clip_ndvi(ndvi_copy)

In [ ]:
clipped_ndvi.rio.set_spatial_dims(x_dim='x', y_dim='y', inplace=True)
clipped_ndvi.rio.write_crs("EPSG:32648", inplace=True)  # Ví dụ: thiết lập CRS là WGS84

# Xuất DataArray thành file TIFF
output_path = "../output/fourier/lt_pred_cut.tif"
clipped_ndvi.rio.to_raster(output_path)

In [ ]:
#Hiển thị sau khi đã dự đoán
for i in range(13):
    ndvi_plot = clipped_ndvi.sel(band=i+1)
    plt.figure(figsize=(10, 8))
    img = plt.imshow(ndvi_plot, vmin=-1, vmax=1)
    plt.title(f"NDVI sau lọc - Band {i}")
    plt.colorbar(img, label="NDVI")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()
